In [14]:
from mixed_naive_bayes import MixedNB
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tqdm import trange

In [15]:
CATEGORICAL_FILE = r"../data/drug_consumption_transformed.csv"
NUMERICAL_FILE = r"../data/drug_consumption_numeric.csv"

OUTPUT_CAT = r"../data/nb/pred_cat.csv"
OUTPUT_NUM = r"../data/nb/pred_num.csv"

In [ ]:
df_cat = pd.read_csv(CATEGORICAL_FILE)
X_cat = df_cat.drop(columns = "choc")
y_cat = df_cat["choc"]

df_num = pd.read_csv(NUMERICAL_FILE)
X_num = df_num.drop(columns = "choc")
y_num = df_num["choc"]

n = len(df_cat)

categorical_columns_cat = [0, 1, 2, 3, 4]
categorical_columns_num = [1, 3, 4]

In [ ]:
# MixedNB function requires categorical data to be assigned numbers
le = LabelEncoder()

true_values = df_cat["choc"].str[2].astype(int)

for i in categorical_columns_cat:
    col_name = X_cat.columns[i]
    le.fit(X_cat[col_name])
    X_cat[col_name] = le.transform(X_cat[col_name])
y_cat = y_cat.str[2].astype(int)

for i in categorical_columns_num:
    col_name = X_num.columns[i]
    le.fit(X_num[col_name])
    X_num[col_name] = le.transform(X_num[col_name])
y_num = y_num.str[2].astype(int)

In [17]:
alphas = 10**np.arange(-2,2.05,0.05)

In [19]:
pred_cat = np.zeros((len(alphas), n))
for i in trange(n):
    X_cat_train = X_cat.drop(i)
    X_cat_test = X_cat.iloc[[i]]
    y_cat_train = np.delete(y_cat, i)
    y_cat_test = y_cat[i]

    for j, alpha in enumerate(alphas):
        nb_cat = MixedNB(categorical_features=categorical_columns_cat, alpha = alpha)
        nb_cat.fit(X_cat_train.values, y_cat_train)
        pred = nb_cat.predict(X_cat_test.values)[0]
        pred_cat[j, i] = pred

np.savetxt(OUTPUT_CAT, pred_cat, delimiter=",", fmt='%s')

100%|██████████| 1885/1885 [04:29<00:00,  7.00it/s]


In [20]:
pred_num = np.zeros((len(alphas), n))
for i in trange(n):
    X_num_train = X_num.drop(i)
    X_num_test = X_num.iloc[[i]]
    y_num_train = np.delete(y_num, i)
    y_num_test = y_num[i]

    for j, alpha in enumerate(alphas):
        nb_num = MixedNB(categorical_features=categorical_columns_num, alpha = alpha)
        nb_num.fit(X_num_train.values, y_num_train)
        pred = nb_num.predict(X_num_test.values)[0]
        pred_num[j, i] = pred

np.savetxt(OUTPUT_NUM, pred_num, delimiter=",", fmt='%s')

100%|██████████| 1885/1885 [03:31<00:00,  8.93it/s]
